In [20]:
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
import os
current_workdir = os.getcwd()
output_dir = os.path.join(current_workdir, 'outputs')

In [4]:
# -------------------- 1. 加载 CIFAR-10 数据集 --------------------
# 使用训练集（包含 50000 张图像）进行统计和可视化
transform = transforms.Compose([
    transforms.ToTensor(),  # 将 PIL 图像转为 [0,1] 的 Tensor，同时自动归一化到 [0,1]
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=transform)
# 为了计算均值和标准差，我们还需要原始像素值（0-255），但 ToTensor 已归一化，所以我们可以直接使用 tensor 值
# 但是标准计算时通常使用 [0,1] 范围，这里按常见做法使用 [0,1] 计算
# 为了方便显示和亮度直方图，我们也可以额外加载原始 PIL 图像

# 加载原始 PIL 图像（不应用 transform），用于亮度计算和拼图
pil_transform = transforms.Compose([])  # 恒等变换
pil_trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=False, transform=pil_transform)

In [ ]:

# -------------------- 2. 统计基本信息 --------------------
# 类别名称
classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']

# 类别数量（每个类别的样本数）
labels = trainset.targets  # 所有标签
class_counts = [labels.count(i) for i in range(10)]
print("每个类别的样本数:")
for i, cls in enumerate(trainset.classes):
    print(f"{cls}: {class_counts[i]}")

# 图像尺寸和通道数（CIFAR-10 固定为 32x32 RGB）
img, _ = trainset[0]  # 获取一个样本
height, width = img.shape[1], img.shape[2]  # Tensor 形状 (C, H, W)
channels = img.shape[0]
print(f"\n图像尺寸: {height} x {width}, 通道数: {channels}")

# dtype
print(f"数据 dtype (Tensor): {img.dtype}")  # torch.float32，因为 ToTensor 转换为 float
# 原始 PIL 图像的 dtype 是 uint8
pil_img, _ = pil_trainset[0]
print(f"原始 PIL 图像 dtype: {np.array(pil_img).dtype}")

# 计算整体均值和标准差（按通道）
# 将整个训练集的所有图像堆叠成一个大 Tensor，形状 (N, C, H, W)
all_images = torch.stack([trainset[i][0] for i in range(len(trainset))])  # (50000, 3, 32, 32)
mean = all_images.mean(dim=(0, 2, 3))  # 按通道求均值
std = all_images.std(dim=(0, 2, 3))    # 按通道求标准差
print(f"\n整体均值 (R, G, B): {mean.tolist()}")
print(f"整体标准差 (R, G, B): {std.tolist()}")

In [ ]:

# -------------------- 3. 绘制类别分布条形图 --------------------
plt.figure(figsize=(10, 6))
plt.bar(classes, class_counts, color='skyblue')
plt.xlabel('classes', fontsize=14)
plt.ylabel('sample amount', fontsize=14)
plt.title('CIFAR-10 trainset_class_distribution', fontsize=14)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'trainset_class_distribution.png'))
plt.show()
print("类别分布图已保存为 cifar10_class_distribution.png")

In [ ]:

# -------------------- 4. 绘制亮度直方图 --------------------
# 亮度定义为 RGB 三个通道的平均值（灰度化）
brightness_list = []
for pil_img, _ in pil_trainset:
    # 将 PIL 图像转为 numpy 数组 (H, W, C)
    arr = np.array(pil_img)  # 0-255 uint8
    gray = arr.mean(axis=2)  # (H, W) 灰度值
    brightness_list.extend(gray.flatten())  # 展平所有像素的亮度

plt.figure(figsize=(10, 6))
plt.hist(brightness_list, bins=256, range=(0, 255), color='gray', alpha=0.7)
plt.xlabel('brightness (0-255)')
plt.ylabel('px amount')
plt.title('CIFAR-10 trainset_brightness_histogram')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('cifar10_brightness_histogram.png')
plt.show()
print("亮度直方图已保存为 cifar10_brightness_histogram.png")

In [ ]:

# -------------------- 5. 生成每类样本拼图 --------------------
# 每类选取 16 张样本，拼成 4x4 网格
n_samples_per_class = 16
grid_rows = 4
grid_cols = 4

# 用于存储每个类别的样本图像（原始 PIL 格式）
class_samples = {i: [] for i in range(10)}
# 收集每类的前 n_samples_per_class 个样本
for idx, (img_pil, label) in enumerate(pil_trainset):
    if len(class_samples[label]) < n_samples_per_class:
        class_samples[label].append(img_pil)
    # 如果所有类别都收集够了，提前退出
    if all(len(lst) == n_samples_per_class for lst in class_samples.values()):
        break

# 为每个类别生成拼图
for cls in range(10):
    fig, axes = plt.subplots(grid_rows, grid_cols, figsize=(6, 6))
    fig.suptitle(f'Class: {classes[cls]}', fontsize=16)
    for i in range(grid_rows):
        for j in range(grid_cols):
            idx = i * grid_cols + j
            img = class_samples[cls][idx]
            axes[i, j].imshow(img)
            axes[i, j].axis('off')
    plt.tight_layout()
    plt.savefig(f'cifar10_class_{cls}_{classes[cls]}_collage.png')
    plt.close()  # 关闭图形，防止内存堆积

print(f"每类 {n_samples_per_class} 张样本拼图已保存为 cifar10_class_*_collage.png")

# 也可以生成一张大图包含所有类别的拼图（可选）
# 这里不做，但可以展示